<a href="https://colab.research.google.com/github/ISPC-PROYECTOS/CienciaDatosABP/blob/main/Data_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importamos el dataset de Kaggle, buscamos el archivo CSV

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("lucalullo/global-ghg-gas-emissions-by-country-1950-2024")
print("Ruta del dataset:", path)

archivos = os.listdir(path)
print("Archivos descargados:", archivos)

100%|██████████| 299k/299k [00:00<00:00, 69.9MB/s]

Extracting files...
Ruta del dataset: /root/.cache/kagglehub/datasets/lucalullo/global-ghg-gas-emissions-by-country-1950-2024/versions/1
Archivos descargados: ['emissioni-gas-serra-globali-per-paese-1950-2024.csv']


Buscamos el primer archivo que termine en .csv y unimos la ruta de la carpeta con el nombre del archivo. Luego lo cargamos en pandas y verificamos que se haya cargado.

In [ ]:
import pandas as pd
import plotly.express as px

# Buscamos el primer archivo que termine en '.csv'
nombre_csv = [archivo for archivo in archivos if archivo.endswith('.csv')][0]

# Unimos la ruta de la carpeta con el nombre del archivo
ruta_completa = os.path.join(path, nombre_csv)

# Lo cargamos en pandas
df = pd.read_csv(ruta_completa)

# Vemos las primeras 5 filas para confirmar que todo salió bien
df.head()

,country,year,iso_code,total_ghg,ghg_per_capita
0,Afghanistan,1950,AFG,19.868742,2.555078
1,Afghanistan,1951,AFG,21.069101,2.673967
2,Afghanistan,1952,AFG,22.094320,2.766014
3,Afghanistan,1953,AFG,23.255630,2.872235
4,Afghanistan,1954,AFG,24.250988,2.954572


# **Primer análisis del dataset**
Este conjunto de datos contiene las emisiones de gases de efecto invernadero a nivel nacional desde 1950 hasta 2024.

El conjunto de datos se deriva del conjunto de datos de emisiones de CO₂ y gases de efecto invernadero publicado por Our World in Data, que integra varias fuentes científicas ampliamente utilizadas en la investigación climática.

Cada observación representa un registro de un país y un año, y abarca 199 países de todo el mundo.

Incluye información sobre:
- Emisiones totales de gases de efecto invernadero
- Emisiones de gases de efecto invernadero per cápita

**Variables**

- Country → nombre del país
- Year → año de observación
- Iso_code → Código de país ISO
- Total_ghg → emisiones totales de gases de efecto invernadero expresadas en millones de toneladas de CO₂ equivalente
- Ghg_per_capita → emisiones de gases de efecto invernadero por persona, expresadas en toneladas de CO₂ equivalente per cápita.

El dataset posee un total de 14.925 registros

**Fuentes**

- Dataset “Global GHG gas emissions by country (1950–2024)” publicado por Luca Lullo en Kaggle.
- Datos originales derivados de Our World in Data, que integran:

   - Global Carbon Project - Global Carbon Budget
   - PRIMAP-hist greenhouse gas emissions database
   - United Nations World Population Prospects
   - Gapminder population series


# **Análisis de datos inicial**
Vamos a explorar el df para entender su estructura, identificar valores nulos, duplicados y obtener estadísticas descriptivas.

In [ ]:
# 1. Información general del DataFrame y tipos de datos
print('Información general del DataFrame:')
df.info()

# 2. Conteo de valores nulos por columna
print('\nConteo de valores nulos por columna:')
display(df.isnull().sum())

Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14925 entries, 0 to 14924
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         14925 non-null  object 
 1   year            14925 non-null  int64  
 2   iso_code        14925 non-null  object 
 3   total_ghg       14475 non-null  float64
 4   ghg_per_capita  14475 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 583.1+ KB

Conteo de valores nulos por columna:


,0
country,0
year,0
iso_code,0
total_ghg,450
ghg_per_capita,450


El dataset muestra un total de 14.925 registros distribuidos en cinco columnas. Las variables categóricas y la temporal están completas, pero las métricas de emisiones (total_ghg y ghg_per_capita) presentan 450 valores faltantes cada una (un 3 % del total). Esto indica que, aunque la mayoría de este dataset es consistente, será necesario aplicar técnicas de tratamiento de datos faltantes para garantizar la validez de futuros análisis.

In [ ]:
# 3. Conteo de filas duplicadas
num_duplicados = df.duplicated().sum()
print(f'Número de filas duplicadas: {num_duplicados}')

if num_duplicados > 0:
    print('\nEjemplo de filas duplicadas:')
    display(df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()).head()) # Muestra algunas filas duplicadas

Número de filas duplicadas: 0


El análisis de duplicados muestra que el dataset no tiene filas repetidas, confirmando que son registros únicos y no será necesario aplicar técnicas de limpieza relacionadas con duplicados.

In [ ]:
# 4. Parámetros estadísticos descriptivos para columnas numéricas
print('\nParámetros estadísticos descriptivos:')
display(df.describe())


Parámetros estadísticos descriptivos:


,year,total_ghg,ghg_per_capita
count,14925.000000,14475.000000,14475.000000
mean,1987.000000,188.028820,9.259861
std,21.649436,706.305423,12.160517
min,1950.000000,-9.218572,-5.197811
25%,1968.000000,7.289340,2.858160
50%,1987.000000,34.407463,5.909035
75%,2006.000000,99.218967,11.049553
max,2024.000000,14107.006836,369.705292


Las columnas numéricas muestran que la variable "year" abarca desde 1950 hasta 2024. Las variables de emisiones (total_ghg y ghg_per_capita) muestran una gran variabilidad entre países y años; estos parámetros permiten identificar tendencias centrales y la dispersión de los datos.

In [ ]:
# 5. Parámetros estadísticos descriptivos para columnas categóricas (objetos)
print('\nParámetros estadísticos descriptivos para columnas categóricas:')
display(df.describe(include='object'))


Parámetros estadísticos descriptivos para columnas categóricas:


,country,iso_code
count,14925,14925
unique,199,199
top,Afghanistan,AFG
freq,75,75


El dataset muestra 199 países distintos. Cada país cuenta con aproximadamente 75 registros, lo que indica la presencia de múltiples observaciones por país a lo largo del rango temporal.

#**Limpieza y estandarización del dataset**
Verificamos si es necesario aplicar algun método de limpieza o estandarización para poder trabajar con los datos.

In [ ]:
#Quitamos los espacios iniciales y finales, y cambiamos los objetos a minúsculas
df['country'] = df['country'].str.strip().str.lower()
df['iso_code'] = df['iso_code'].str.strip().str.lower()

#Listamos los países para verificar si alguno se encuentra escrito de diferente manera
columnas_anormalizar = ["country", "iso_code"]
for columna in columnas_anormalizar:
    print("------")
    print(df[columna].value_counts(dropna=False).to_string())


------
country
afghanistan                         75
albania                             75
algeria                             75
andorra                             75
angola                              75
antigua and barbuda                 75
argentina                           75
armenia                             75
australia                           75
austria                             75
azerbaijan                          75
bahamas                             75
bahrain                             75
bangladesh                          75
barbados                            75
belarus                             75
belgium                             75
belize                              75
benin                               75
bhutan                              75
bolivia                             75
bosnia and herzegovina              75
botswana                            75
brazil                              75
brunei                              75
bulgaria  

El dataset no posee ni nombre del país ni código ISO que se encuentre escrito de una manera diferente.

En las columnas numéricas, el año de observación es de tipo int64 lo cual es correcto ya que todos son valores enteros. Las columnas de emisión total y de emisión per capita son valores float, lo cual tambien corresponde a los valores contenidos. No es necesaria transformación de datos.

Trabajamos los valores nulos de emisión total y de emisión per capita. Para esto procederemos primero a identificar los países que no poseen datos registrados y los eliminamos.
Los motivos para ésto son:


1.   Cero información: Al tener un 100% de nulos, no aportan nada a promedios, tendencias o sumatorias. Son "filas vacías" para el análisis de emisiones.
2.   Imputación imposible: No podemos "rellenar" los datos (imputación) basándonos en años anteriores o posteriores porque no existe ningún punto de referencia en todo el histórico de esos países.
3.   Limpieza estadística: Si calculamos, por ejemplo, el "Promedio de emisiones por país", tener estos países en la lista nos obligaría a estar filtrando nulos constantemente para evitar errores o resultados NaN.
4.   Impacto mínimo: Eliminar estos 6 países solo reduce el dataset en 450 filas de un total de casi 15,000 (Aproximadamente 3%), pero mejora significativamente la "salud" de las estadísticas.



In [ ]:
# Identificamos países que tienen TODOS sus valores de total_ghg como nulos
paises_sin_datos = df.groupby('country')['total_ghg'].apply(lambda x: x.isnull().all())
nombres_paises_vacios = paises_sin_datos[paises_sin_datos].index.tolist()

print(f"Países eliminados por falta total de datos: {nombres_paises_vacios}")

# Filtramos el DataFrame original para conservar solo los países que SÍ tienen datos
df_limpio = df[~df['country'].isin(nombres_paises_vacios)].copy()

# Verificamos resultados
print(f"Registros originales: {len(df)}")
print(f"Registros tras la limpieza: {len(df_limpio)}")

Países eliminados por falta total de datos: ['kosovo', 'marshall islands', 'monaco', 'palestine', 'san marino', 'vatican']
Registros originales: 14925
Registros tras la limpieza: 14475


Vamos a agregar una columna de cantidad estimada de población, basándonos en los datos de emisión total y emisión per capita.
En este cálculo, como las cantidades en emisión total estan expresadas en millones de toneladas de CO₂ equivalente, el resultado será a su vez en millones de habitantes.

In [ ]:
#Creamos la nueva columna "poblacion_en_millones" y hacemos el cálculo
df_limpio['poblacion_en_millones'] = df_limpio['total_ghg'] / df_limpio['ghg_per_capita']

# Verificamos los resultados
print(df_limpio[['country', 'year', 'poblacion_en_millones']].head())

       country  year  poblacion_en_millones
0  afghanistan  1950               7.776180
1  afghanistan  1951               7.879343
2  afghanistan  1952               7.987784
3  afghanistan  1953               8.096703
4  afghanistan  1954               8.207954


#**Dataset final**

In [ ]:
#Exportamos el dataset resultante luego de la limpieza y estandarización
df_limpio.to_csv('dataset_final.csv', index=False)


#**ENTREGA FINAL**

##**Analisis descriptivo**


###1.   Entendimiento del contexto y del dataset

El conjunto de datos contiene registros anuales sobre emisiones de gases de efecto invernadero y población a nivel nacional, abarcando desde $1950$ hasta $2024$. La unidad de observación es el "país-año". Técnicamente, el archivo presenta una estructura de panel balanceado con $193$ países seguidos ininterrumpidamente durante $75$ años, lo que da un total de $14475$ observaciones. Contamos con tres variables numéricas clave:

* ***total\_ghg:*** Emisiones totales de gases de efecto invernadero del país en ese año.

* ***poblacion\_en\_millones:*** Población total del territorio.

* ***ghg\_per\_capita:*** Emisiones por habitante.

###2.   Estructura y calidad de los datos


  *   **Integridad:** El dataset no presenta valores faltantes (nulos) en ninguna de sus columnas ni registros duplicados.

  *   **Anomalías detectadas:** Se identificaron $104$ registros con valores de emisiones negativas (***total\_ghg < 0***), concentrados principalmente en la década de $1950$ en islas pequeñas o naciones con baja densidad industrial (como Antigua y Barbuda o Guinea Ecuatorial). Esto no representa un error de software, sino que indica periodos donde la absorción natural de carbono del territorio (bosques y suelo) fue mayor que sus emisiones industriales, actuando como un "sumidero neto". En los años recientes (especialmente en $2024$), esta condición desaparece por completo; el valor mínimo registrado es positivo ($0.0198$).

###3. Distribución estadística general

* **Medias y Medianas (Tendencias centrales):** La media (promedio) histórica de las emisiones totales por país es de $188.02$ millones de toneladas. Sin embargo, la mediana (el valor que divide la muestra exactamente a la mitad) es de apenas $34.40$. Esta enorme diferencia entre la media y la mediana indica una distribución profundamente asimétrica: la gran mayoría de los países emiten muy poco, pero el promedio se ve arrastrado hacia arriba por un grupo reducido de países altamente contaminantes.

* **Máximos y Mínimos (Dispersión):** La dispersión de los datos es extrema. El valor máximo histórico registrado de emisiones totales en un solo año alcanza las $14107.00$ millones de toneladas, lo cual representa una brecha abismal frente al resto. Por otro lado, el valor mínimo es de $-9.21$, lo que estadísticamente se considera un valor atípico (outlier) que, como se explicó en la calidad de los datos, corresponde a países que actúan como sumideros netos de carbono.

* **Emisiones per cápita:** En cuanto a las emisiones por persona, el promedio histórico se sitúa en $9.25$ unidades, con una mediana de $5.90$. El registro máximo de esta variable alcanza un pico extremo de $369.70$ unidades por habitante, demostrando que ciertos territorios tienen un nivel de contaminación individual completamente desproporcionado respecto a la media global.


## **Análisis visual y exploración de patrones**

###1.  Análisis individual de las variables

Para entender cómo cambió el mundo en estos $75$ años, comparamos los agregados del inicio y el final de la serie temporal:

*   *Población mundial:* En $1950$, la población global sumada en este dataset era de $2487$ millones de habitantes. Para $2024$, este número se triplicó críticamente, alcanzando los $8146$ millones.

In [ ]:
# Agrupamos datos a nivel global por año
global_df = df_limpio.groupby('year').agg(
    Emisiones_Totales=('total_ghg', 'sum'),
    Poblacion_Total=('poblacion_en_millones', 'sum')
).reset_index()

global_df['Emisiones_Por_Habitante'] = global_df['Emisiones_Totales'] / global_df['Poblacion_Total']

# Gráfico 1: Evolución de Emisiones Globales Totales
fig1 = px.line(
    global_df,
    x='year',
    y='Emisiones_Totales',
    title='Evolución de las Emisiones Globales de GEI (1950-2024)',
    labels={'year': 'Año', 'Emisiones_Totales': 'Gases de Efecto Invernadero (Unidades)'},
    template='plotly_white'
)
fig1.update_traces(line_color='#e74c3c', line_width=3)
fig1.show()

*   *Emisiones totales:* En $1950$, el planeta generaba $17039$ unidades de gases de efecto invernadero. En $2024$, el volumen total ascendió a $54411$ unidades. Las emisiones globales crecieron un $219\%$.

In [ ]:
# Gráfico 2: Evolución de Emisiones por Persona (Muestra el pico en 1970)
fig2 = px.line(
    global_df,
    x='year',
    y='Emisiones_Por_Habitante',
    title='Promedio Mundial de Emisiones por Habitante (1950-2024)',
    labels={'year': 'Año', 'Emisiones_Por_Habitante': 'Emisiones por Persona'},
    template='plotly_white'
)
fig2.update_traces(line_color='#2c3e50', line_width=3)
fig2.show()

*   *Indicador por persona:* A pesar del alarmante incremento total, al promediar las emisiones globales por cada habitante del planeta, el pico máximo de la historia no ocurrió en la actualidad, sino alrededor de el año $1970$, cuando se alcanzó un promedio de $7.83$ unidades por persona. Desde entonces, este indicador ha descendido gradualmente hasta situarse en $6.68$ en $2024$. Esto demuestra que, de manera general, la humanidad produce menos gases por individuo hoy que hace 50 años.

###2. Análisis cruzado: concentración y desacoplamiento

Al cruzar la evolución de la población con el volumen de emisiones, se revelan dos fenómenos que transforman por completo la interpretación del problema climático:

***A. La concentración extrema de las emisiones***

Existe una profunda asimetría en la distribución de la contaminación. Las emisiones no se reparten de manera equitativa según la población de los países, y esta brecha se ha intensificado con el tiempo:

* En $1950$: Los 5 países con mayores emisiones (Estados Unidos, China, Rusia, India y Brasil) concentraban el $44.87\%$ de la contaminación mundial y albergaban al $48.29\%$ de la población humana. Había un equilibrio relativo entre masa poblacional y emisiones.


In [ ]:
# Filtramos año 2024 y ordenar
df_2024 = df_limpio[df_limpio['year'] == 2024].sort_values(by='total_ghg', ascending=False)

# Separamos Top 10 y el resto
top_10 = df_2024.head(10).copy()
resto_emisiones = df_2024.iloc[10:]['total_ghg'].sum()

# Creamos un registro para "Resto del Mundo" para simplificar la visualización
resto_df = pd.DataFrame([{
    'country': 'Resto del Mundo (183 países)',
    'total_ghg': resto_emisiones,
    'year': 2024,
    'iso_code': 'REST',
    'ghg_per_capita': 0,
    'poblacion_en_millones': 0
}])

df_concentracion = pd.concat([top_10, resto_df])

# Gráfico 3: Distribución de emisiones en 2024
fig3 = px.bar(
    df_concentracion,
    x='country',
    y='total_ghg',
    title='Distribución de Emisiones en 2024: Top 10 vs Resto del Mundo',
    labels={'country': 'País / Región', 'total_ghg': 'Emisiones Totales'},
    color='country',
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)
fig3.update_layout(showlegend=False)
fig3.show()

* En $2024$: Los mismos 5 países (con variaciones en sus posiciones internas) ahora concentran el $55.06\%$ de todas las emisiones del planeta, pero su participación poblacional bajó al $43.85\%$. Es decir, hoy en día un grupo más reducido de la población mundial genera una porción sustancialmente mayor de la contaminación global. Solo el Top 10 de países acumula el $65.25\%$ del impacto ecológico actual.

***B. El fenómeno del desacoplamiento (¿Más gente implica más contaminación?)***

El análisis histórico desmitifica la idea de que el crecimiento de la población obliga un aumento proporcional de las emisiones. Al evaluar el comportamiento desde la adopción de políticas climáticas globales (tomando como base el año $1990$ frente a $2024$), se observan tres realidades completamente distintas.

Para ilustrar visualmente estos contrastes de manera clara, el siguiente gráfico tomará como casos de estudio a los líderes indiscutidos de cada modelo: Estados Unidos y China.

In [ ]:
# Filtramos los dos casos de estudio principales desde 1990
casos_estudio = df_limpio[df_limpio['country'].isin(['united states', 'china', 'russia', 'india']) & (df_limpio['year'] >= 1990)]

# Gráfico 4: Trayectoria de Población vs Emisiones
fig4 = px.line(
    casos_estudio,
    x='poblacion_en_millones',
    y='total_ghg',
    color='country',
    hover_data=['year'],
    title='Trayectoria de Desacoplamiento (EE. UU.) vs Acoplamiento (China) desde 1990',
    labels={'poblacion_en_millones': 'Población (Millones)', 'total_ghg': 'Emisiones Totales de GEI'},
    template='plotly_white'
)

# Añadimos puntos individuales para identificar los años al pasar el cursor
fig4.update_traces(mode='lines+markers')
fig4.show()

1. **Desacoplamiento absoluto (El caso de las economías maduras):**

* **Estados Unidos:** Entre $1990$ y $2024$, su población creció un $36\%$ (pasando de $253$ a $345$ millones), pero sus emisiones totales disminuyeron un $6.7\%$ (de $6489$ a $6054$ unidades). Como resultado, la contaminación por habitante cayó drásticamente de $25.61$ a $17.52$.

* **Rusia (caso complementario):** Mostró una tendencia similar, reduciendo sus emisiones de $3462$ a $2729$ unidades en el mismo periodo. En estos países, el crecimiento demográfico se separó por completo del daño ambiental.

---

2. **Acoplamiento acelerado (El motor industrial emergente):**

* **China:** Entre $1990$ y $2024$, su población aumentó un aceptable $23\%$ (de $1153$ a $1419$ millones), pero sus emisiones se triplicaron de forma masiva un $235\%$ (pasando de $4205$ a $14107$ unidades). Su indicador por persona subió de $3.65$ a $9.94$.

* **India (caso complementario):** Vivió un proceso similar; su población creció un $67\%$, pero sus emisiones se dispararon un $179\%$. En esta región, la industrialización superó por completo al ritmo del crecimiento demográfico.

---

3. **El resto del mundo (Desacoplamiento relativo):**

* Si aislamos a los 5 grandes emisores y analizamos en conjunto a los $188$ países restantes, descubrimos una tendencia positiva: entre $1990$ y $2024$, la población de este bloque creció un notable $66\%$ (de $2746$ a $4574$ millones), pero sus emisiones conjuntas solo aumentaron un $23\%$. Esto provocó que el promedio de emisiones por habitante en el resto del mundo cayera de $7.20$ a $5.34$.

###3. Síntesis de hallazgos claves

* **Dominio de China:** Para el año $2024$, China se convirtió en el emisor más grande con diferencia ($14107$ unidades), duplicando al segundo lugar (Estados Unidos con $6054$).

* **La ilusión del promedio global:** Aunque el promedio de emisiones por persona a nivel mundial viene bajando desde $1970$, esto ocurre porque la gran mayoría de las naciones ($188$ países) están logrando crecer demográficamente sin disparar sus emisiones a la misma velocidad. Sin embargo, este esfuerzo colectivo se ve opacado en los indicadores globales debido al crecimiento industrial acelerado de potencias concentradas como China e India.

###4. Conclusiones

El análisis descriptivo demuestra que los datos son técnicamente limpios, consistentes y sumamente explicativos. La relación entre población y gases de efecto invernadero no es lineal ni uniforme: está fragmentada por la realidad económica de cada país.

1. **El mito de la culpabilidad demográfica global:** La narrativa común de que el aumento poblacional es el causante directo del incremento de emisiones es falsa a nivel desagregado. Si el crecimiento poblacional fuera el motor principal, el bloque de 188 países (excluyendo al Top 5) no habría registrado una caída en sus emisiones por habitante de $7.20$ a $5.34$ entre 1990 y 2024, un periodo donde sumaron más de 1800 millones de personas. El crecimiento demográfico no dicta las emisiones; lo hace la estructura de la matriz energética y el modelo industrial adoptado.

2. **La ilusión de la acción climática global homogénea:** Los esfuerzos de mitigación o los acuerdos internacionales carecen de impacto real si se analizan como un promedio de 193 países. El destino del balance atmosférico terrestre está centralizado. Un aumento del 1% en las emisiones de China equivale a la producción anual completa de decenas de países pequeños. Por lo tanto, tratar el cambio climático como un problema estadísticamente homogéneo distorsiona la realidad operativa: es un problema de concentración industrial extrema.

3. **La exportación de la huella de carbono:** El "desacoplamiento absoluto" observado en Estados Unidos (donde la población sube un 36% y las emisiones bajan un 6.7%) no se debe únicamente a una transición interna a energías limpias. Coincide cronológicamente con el "acoplamiento acelerado" de China (donde las emisiones se dispararon un 235%). Esto sugiere un desplazamiento de la manufactura pesada y la producción de bienes intensivos en carbono desde las economías de consumo hacia las economías de producción. El norte global consume lo que el este asiático produce emitiendo gases.

